# 01 — Data Exploration: Jena Climate (Temperature)

**Project:** Time-Series Data Analysis Through a Natural HCI Interface  
**Notebook goal:** load, document, and explore the temperature time-series dataset before building the pipeline.

> Reminder of the core principle: **computation = code (truth), LLM = explanation only.** This notebook is part of the *computation* side — understanding the data.

## Contents
1. Dataset description
2. Load the data
3. Parse timestamps & set the time index
4. Inspect structure (shape, types, preview)
5. Missing data check
6. Descriptive statistics
7. Temperature over time
8. Multi-sensor relationships (correlation)
9. Next steps

## 1. Dataset description

**Jena Climate (2009–2016)** — weather readings recorded **every 10 minutes** at the Max Planck Institute, Jena. ~420,000 rows, 14 sensor columns. Time-indexed and multi-sensor — ideal for behaviour analysis, anomaly detection, and multi-sensor reasoning.

| Column | Meaning |
| --- | --- |
| `Date Time` | timestamp (every 10 min) |
| `p (mbar)` | atmospheric pressure |
| `T (degC)` | **air temperature** (primary target) |
| `Tpot (K)` | potential temperature |
| `Tdew (degC)` | dew-point temperature |
| `rh (%)` | relative humidity |
| `VPmax (mbar)` | saturation vapour pressure |
| `VPact (mbar)` | actual vapour pressure |
| `VPdef (mbar)` | vapour pressure deficit |
| `sh (g/kg)` | specific humidity |
| `H2OC (mmol/mol)` | water vapour concentration |
| `rho (g/m**3)` | air density |
| `wv (m/s)` | wind speed |
| `max. wv (m/s)` | max wind speed |
| `wd (deg)` | wind direction |

## 2. Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# data/ lives at the repo root: code/data/  (this notebook is in code/v1/notebooks/)
DATA_PATH = "../../data/jena_climate_2009_2016.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 3. Parse timestamps & set the time index

The `Date Time` column is in `DD.MM.YYYY HH:MM:SS` format. We convert it to a proper datetime index so every later step (resampling, rolling stats, decomposition) works correctly.

In [ ]:
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df = df.set_index("Date Time").sort_index()

print("Time range:", df.index.min(), "->", df.index.max())
print("Most common interval:", df.index.to_series().diff().mode()[0])

## 4. Inspect structure

In [ ]:
df.info()

## 5. Missing data check

Counting missing values is something the **query engine** must do exactly later — here we just get an overview.

In [ ]:
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "None — dataset is complete.")

## 6. Descriptive statistics (temperature focus)

In [ ]:
temp = df["T (degC)"]
print(f"Temperature  min={temp.min():.2f}  max={temp.max():.2f}  "
      f"mean={temp.mean():.2f}  std={temp.std():.2f} degC")
print("Coldest reading at:", temp.idxmin())
print("Warmest reading at:", temp.idxmax())
df.describe().T

## 7. Temperature over time

Raw 10-minute data is dense; we also resample to **daily mean** to see the seasonal pattern clearly.

In [ ]:
daily_temp = temp.resample("D").mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily_temp.index, daily_temp.values, linewidth=0.8)
ax.set_title("Daily mean air temperature (Jena, 2009–2016)")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (degC)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Multi-sensor relationships

Correlation across sensors is the basis for the **multi-sensor reasoning** novelty (e.g. temperature vs humidity vs pressure).

In [ ]:
corr = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Sensor correlation matrix")
plt.tight_layout()
plt.show()

print("Strongest correlations with temperature:")
print(corr["T (degC)"].drop("T (degC)").sort_values(ascending=False))

## 9. Next steps

- **Phase 1 — Preprocessing:** move the cleaning/indexing logic above into `src/preprocess.py`.
- **Phase 2 — Analysis:** rolling stats + STL decomposition in `src/analysis.py`.
- **Phase 3 — Anomaly detection:** z-score baseline, then Isolation Forest.
- **Phase 5 — Query engine:** turn the exact stats above (min/max/mean/missing/correlation) into deterministic functions — the truth layer.